# LTR DEMO BY MADHAV A NAIR

## Phase 1 : data setup

In [1]:
from ltr.client import OpenSearchClient
from ltr.index import rebuild
from ltr.helpers.movies import indexable_movies
from ltr import download

client = OpenSearchClient()
corpus='http://es-learn-to-rank.labs.o19s.com/tmdb.json'
download([corpus], dest='data/')

movies = indexable_movies(movies='data/tmdb.json')
rebuild(client, index='tmdb', doc_src=movies)

http://localhost:9201/_ltr; <OpenSearch([{'host': 'localhost', 'port': 9201}])>
data/tmdb.json already exists
Index tmdb already exists. Use `force = True` to delete and recreate


In [2]:
import requests

host = client.get_host()
port = 9201

url = f"http://{host}:{port}"

response = requests.get(url)
print(response.status_code)
print(response.text)
print(response.url)

200
{
  "name" : "opensearch-node1",
  "cluster_name" : "opensearch-cluster",
  "cluster_uuid" : "XJJLwlH-SvKEyLgFg34phg",
  "version" : {
    "distribution" : "opensearch",
    "number" : "2.5.0",
    "build_type" : "tar",
    "build_hash" : "b8a8b6c4d7fc7a7e32eb2cb68ecad8057a4636ad",
    "build_date" : "2023-01-18T23:48:48.981786100Z",
    "build_snapshot" : false,
    "lucene_version" : "9.4.2",
    "minimum_wire_compatibility_version" : "7.10.0",
    "minimum_index_compatibility_version" : "7.0.0"
  },
  "tagline" : "The OpenSearch Project: https://opensearch.org/"
}

http://localhost:9201/


In [3]:
response = requests.get(f"{url}/_ltr")

print(response.status_code)
print(response.text)

200
{"stores":{"_default_":{"store":"_default_","index":".ltrstore","version":2,"counts":{"featureset":1,"model":1}}}}


## Init Default Feature Store
The feature store can be removed by sending a DELETE request to `_ltr` endpoint. This is incase of an old featureset present in index

In [4]:
url = f'http://{host}:{port}/_ltr/'
print(url)
requests.delete(url)

http://localhost:9201/_ltr/


<Response [200]>

To initialize the LTR plugin, issue a PUT request to the `_ltr` endpoint.

In [5]:
print(url)
requests.put(url)
print(response.text)
print(response.status_code)

http://localhost:9201/_ltr/
{"stores":{"_default_":{"store":"_default_","index":".ltrstore","version":2,"counts":{"featureset":1,"model":1}}}}
200


In [6]:
print(url)

http://localhost:9201/_ltr/


## Create Feature Set

A feature set can be created by issuing a PUT to `_ltr/featureset/[feature_name] `

In [7]:
feature_set = {
   "featureset": {
      "features": [
         {
            "name": "title_bm25",
            "params": ["keywords"],
            "template": {
               "match": {
                  "title": "{{keywords}}"
               }
            }
         },
         {
            "name": "overview_bm25",
            "params": ["keywords"],
            "template": {
               "match": {
                  "overview": "{{keywords}}"
               }
            }
         },
         {
            "name": "release_year",
            "params": [],
            "template": {
               "function_score": {
                  "field_value_factor": {
                     "field": "release_year",
                     "missing": 2000
                  },
                  "query": {
                     "match_all": {}
                  }
               }
            }
         }
      ]
   },
   "validation": {
      "index": "tmdb",
      "params": {
         "keywords": "rambo"
      }
   }
}

feature_set_url = f"{url}_featureset/my_feature_set"

response = requests.post(
    feature_set_url,
    json=feature_set,
    headers={"Content-Type": "application/json"}
)

print("Feature URL =", feature_set_url)
print(response.status_code)
print(response.text)

Feature URL = http://localhost:9201/_ltr/_featureset/my_feature_set
201
{"_index":".ltrstore","_id":"featureset-my_feature_set","_version":1,"result":"created","forced_refresh":true,"_shards":{"total":1,"successful":1,"failed":0},"_seq_no":0,"_primary_term":1}


In [8]:
response = requests.get(f"{url}_featureset/my_feature_set")

print(response.status_code)
print(response.text)

200
{"_index":".ltrstore","_id":"featureset-my_feature_set","_version":1,"_seq_no":0,"_primary_term":1,"found":true,"_source":{"name":"my_feature_set","type":"featureset","featureset":{"name":"my_feature_set","features":[{"name":"title_bm25","params":["keywords"],"template_language":"mustache","template":{"match":{"title":"{{keywords}}"}}},{"name":"overview_bm25","params":["keywords"],"template_language":"mustache","template":{"match":{"overview":"{{keywords}}"}}},{"name":"release_year","params":[],"template_language":"mustache","template":{"function_score":{"field_value_factor":{"field":"release_year","missing":2000},"query":{"match_all":{}}}}}]}}}


## PHASE 2  Feature logger

If we have 4 judged documents: 7555,1370, 1369, and 1368 for keywords rambo:

```
doc_id, relevant?, keywords
1368, 1, rambo
1369, 1, rambo
1370, 1, rambo
7555, 0, rambo
```


We need to get feature value for each row.

To do this, we utilize the logging extension to populate the judgment list with features for training. The feature logger script outputs the judgment listwfeatures.txt

## Dummy feature logger script explaining logic

In [24]:
import json
search_with_log = {
  "query": {
    "bool": {
      "filter": [
        {
          "sltr": {
            "_name": "logged_features",
            "featureset": "my_feature_set",
            "params": {
              "keywords": "rambo"
            }
          }
        },
         {
          "terms": {
            "_id": [
              "7555","1370", "1369", "1368"
            ]
          }
        }
      ]
    }
  },
  "ext": {
    "ltr_log": {
      "log_specs": {
        "name": "ltr_features",
        "named_query": "logged_features"
      }
    }
  }
}

search_with_log_url = f'http://{host}:{port}/tmdb/_search'
print(search_with_log_url)

# FIX 1: Use POST instead of GET for complex queries
resp = requests.post(search_with_log_url, json=search_with_log).json()

# FIX 2: Check if results exist before accessing them
print("\n" + "="*80)
print("FEATURE LOGGER RESPONSE (Dummy Script)")
print("="*80)

if resp.get('hits', {}).get('hits'):
    print(f"\n✓ Query returned {len(resp['hits']['hits'])} results")
    print("\nFirst document with logged features:")
    print(json.dumps(resp['hits']['hits'][0], indent=2))
    
    # Show all results summary
    print("\n\nAll Results Summary:")
    for i, hit in enumerate(resp['hits']['hits'], 1):
        doc_id = hit['_source'].get('id', 'N/A')
        title = hit['_source'].get('title', 'N/A')
        score = hit['_score']
        print(f"{i}. Doc ID: {doc_id}, Title: {title}, Score: {score}")
else:
    print("✗ ERROR: No results found")
    print(f"\nTotal hits: {resp.get('hits', {}).get('total', 0)}")
    if 'error' in resp:
        print(f"Server Error: {resp.get('error')}")
    print(f"\nFull response: {json.dumps(resp, indent=2)}")


http://localhost:9201/tmdb/_search

FEATURE LOGGER RESPONSE (Dummy Script)

✓ Query returned 4 results

First document with logged features:
{
  "_index": "tmdb",
  "_id": "1368",
  "_score": 0.0,
  "_source": {
    "id": "1368",
    "title": "First Blood",
    "overview": "When former Green Beret John Rambo is harassed by local law enforcement and arrested for vagrancy, the Vietnam vet snaps, runs for the hills and rat-a-tat-tats his way into the action-movie hall of fame. Hounded by a relentless sheriff, Rambo employs heavy-handed guerilla tactics to shake the cops off his tail.",
    "tagline": "This time he's fighting for his life.",
    "directors": [
      "Ted Kotcheff"
    ],
    "cast": "Sylvester Stallone Richard Crenna Brian Dennehy Bill McKinney Jack Starrett Michael Talbott Chris Mulkey John McLiam Alf Humphreys David Caruso David L. Crowley Don MacKay Charles A. Tamburro David Petersen Craig Huston",
    "genres": [
      "Action",
      "Adventure",
      "Thriller",
   

## Feature logger script using feature logger class 

Populates the judgement list from data/title_judgments.txt with features that are logged and creates a logged_judgements.txt
output would be :
4	qid:1	1:9.655171	2:1767.9543	3:2212.0	4:5.0	5:6.5	6:11.986686	7:0.0	8:11.14768	9:1.0	10:0.0	11:9.655171	12:8.82464	13:1.2858183	14:13.038125	15:9.98079	16:8.82464	17:12.4159355	18:0.0	19:0.0	20:12.0	21:14.0	22:10.381864	23:5.0 # 7555	rambo


In [9]:
from ltr.judgments import judgments_open
from ltr.log import FeatureLogger
from itertools import groupby

# Initialize the Feature Logger
ftr_logger = FeatureLogger(client, index='tmdb', feature_set='my_feature_set')

input_file = 'data/title_judgments.txt'
output_file = 'data/judgments_with_features.txt'

# Log features for each query
with judgments_open(input_file) as judgment_list:
    with open(output_file, 'w') as out_f:
        # Group judgments by query ID
        for qid, query_judgments in groupby(judgment_list, key=lambda j: j.qid):
            judgments_batch = list(query_judgments)
            keywords = judgment_list.keywords(qid)
            
            # Log features for this query
            training_set, discarded = ftr_logger.log_for_qid(
                judgments=judgments_batch,
                qid=qid,
                keywords=keywords
            )
            
            # Write to output file in RankLib format
            for judgment in training_set:
                out_f.write(judgment.toRanklibFormat() + '\n')
            
            if qid % 10 == 0:
                print(f'✓ Logged features for query {qid}')

print(f'✓ Feature logging complete. Output: {output_file}')



Recognizing 40 queries
✓ Logged features for query 10
✓ Logged features for query 20
✓ Logged features for query 30
✓ Logged features for query 40
✓ Feature logging complete. Output: data/judgments_with_features.txt


## Training Set Now...



```
doc_id, relevant?, keywords, title_bm25, overview_bm25
1368, 1, rambo, 0, 11.113943
1369, 1, rambo, 11.657, 10.08
1370, 1, rambo, 9.456, 13.265
7555, 0, rambo, 6.037, 11.114
```



# Train a model

We won't do this here, but we have a different training job script that creates our model to output  

```
cd notebooks/opensearch/tmdb
java -jar data/RankyMcRankFace.jar -train data/title_judgments.txt -save data/model.txt

```

## lambdamart dummy tree creation script given below 

In [26]:
# Generate LambdaMART Model with 10 Trees in XML Format

def generate_lambdamart_model_xml(num_trees=10, learning_rate=0.1):
    """Generate a LambdaMART model with multiple trees in XML format"""
    
    xml_model = """## LambdaMART
## No. of trees = {}
## No. of leaves = 10
## No. of threshold candidates = 256
## Learning rate = {}
## Stop early = 100

<ensemble>
""".format(num_trees, learning_rate)
    
    # Generate trees with varying structures
    for tree_id in range(1, num_trees + 1):
        weight = learning_rate
        xml_model += """	<tree id="{}" weight="{}">
        <split>
            <feature> 2 </feature>
            <threshold> 10.664251 </threshold>
            <split pos="left">
                <feature> 1 </feature>
                <threshold> 0.0 </threshold>
                <split pos="left">
                    <output> {:.4f} </output>
                </split>
                <split pos="right">
                    <feature> 2 </feature>
                    <threshold> 9.502127 </threshold>
                    <split pos="left">
                        <output> {:.4f} </output>
                    </split>
                    <split pos="right">
                        <output> {:.4f} </output>
                    </split>
                </split>
            </split>
            <split pos="right">
                <output> {:.4f} </output>
            </split>
        </split>
    </tree>
""".format(tree_id, weight, 
           -0.5 + (tree_id * 0.05),  # Varying output values per tree
           0.2 + (tree_id * 0.01), 
           0.8 + (tree_id * 0.02),
           1.0 + (tree_id * 0.01))
    
    xml_model += "</ensemble>\n"
    
    return xml_model

# Generate and display the model
model_xml = generate_lambdamart_model_xml(num_trees=10, learning_rate=0.1)
print(model_xml)

# Save to file for use in ltr-tmdb.ipynb
output_path = 'data/latest_model.xml'
with open(output_path, 'w') as f:
    f.write(model_xml)
print(f"\n✓ Model saved to: {output_path}")

## LambdaMART
## No. of trees = 10
## No. of leaves = 10
## No. of threshold candidates = 256
## Learning rate = 0.1
## Stop early = 100

<ensemble>
	<tree id="1" weight="0.1">
        <split>
            <feature> 2 </feature>
            <threshold> 10.664251 </threshold>
            <split pos="left">
                <feature> 1 </feature>
                <threshold> 0.0 </threshold>
                <split pos="left">
                    <output> -0.4500 </output>
                </split>
                <split pos="right">
                    <feature> 2 </feature>
                    <threshold> 9.502127 </threshold>
                    <split pos="left">
                        <output> 0.2100 </output>
                    </split>
                    <split pos="right">
                        <output> 0.8200 </output>
                    </split>
                </split>
            </split>
            <split pos="right">
                <output> 1.0100 </output>
            <

## Uploading a Model
Once features have been logged and training data has been generated, a model can be pushed into OpenSearch.  The following shows what a request to PUT a new model looks like.

In [12]:
model = """## LambdaMART
## No. of trees = 10
## No. of leaves = 10
## No. of threshold candidates = 256
## Learning rate = 0.1
## Stop early = 100

<ensemble>
	<tree id="1" weight="0.1">
		<split>
			<feature> 2 </feature>
			<threshold> 10.664251 </threshold>
			<split pos="left">
				<feature> 1 </feature>
				<threshold> 0.0 </threshold>
				<split pos="left">
					<output> -1.8305741548538208 </output>
				</split>
				<split pos="right">
					<feature> 2 </feature>
					<threshold> 9.502127 </threshold>
					<split pos="left">
						<feature> 1 </feature>
						<threshold> 7.0849166 </threshold>
						<split pos="left">
							<output> 0.23645669221878052 </output>
						</split>
						<split pos="right">
							<output> 1.7593677043914795 </output>
						</split>
					</split>
					<split pos="right">
						<output> 1.9719607830047607 </output>
					</split>
				</split>
			</split>
			<split pos="right">
				<feature> 2 </feature>
				<threshold> 0.0 </threshold>
				<split pos="left">
					<output> 1.3728954792022705 </output>
				</split>
				<split pos="right">
					<feature> 2 </feature>
					<threshold> 8.602512 </threshold>
					<split pos="left">
						<feature> 1 </feature>
						<threshold> 0.0 </threshold>
						<split pos="left">
							<feature> 2 </feature>
							<threshold> 13.815164 </threshold>
							<split pos="left">
								<output> 1.9401178359985352 </output>
							</split>
							<split pos="right">
								<output> 1.99532949924469 </output>
							</split>
						</split>
						<split pos="right">
							<feature> 1 </feature>
							<threshold> 11.085816 </threshold>
							<split pos="left">
								<output> 2.0 </output>
							</split>
							<split pos="right">
								<output> 1.99308180809021 </output>
							</split>
						</split>
					</split>
					<split pos="right">
						<output> 1.9870178699493408 </output>
					</split>
				</split>
			</split>
		</split>
	</tree>
</ensemble>
"""


create_model = {
  "model": {
     "name": "my_model",
     "model": {
         "type": "model/ranklib",
         "definition": model
    }
  }
}

url = 'http://{}:9201/_ltr/_featureset/my_feature_set/_createmodel'.format(host)
print(url)
requests.post(url, json=create_model).json()

http://localhost:9201/_ltr/_featureset/my_feature_set/_createmodel


{'_index': '.ltrstore',
 '_id': 'model-my_model',
 '_version': 1,
 'result': 'created',
 'forced_refresh': True,
 '_shards': {'total': 1, 'successful': 1, 'failed': 0},
 '_seq_no': 1,
 '_primary_term': 1}

In [10]:


# Delete existing model
delete_url = f'http://{host}:9201/_ltr/_featureset/my_feature_set/_model/my_model'
print(requests.delete(delete_url).json())

{'error': 'no handler found for uri [/_ltr/_featureset/my_feature_set/_model/my_model] and method [DELETE]'}


In [11]:


# Re-upload
with open('data/lambdamart_model_10trees.txt', 'r') as f:
    model = f.read()

create_model = {
    "model": {
        "name": "my_model_v2",
        "model": {
            "type": "model/ranklib",
            "definition": model
        }
    }
}

url = f'http://{host}:9201/_ltr/_featureset/my_feature_set/_createmodel'
print(requests.post(url, json=create_model).json())

{'_index': '.ltrstore', '_id': 'model-my_model_v2', '_version': 1, 'result': 'created', 'forced_refresh': True, '_shards': {'total': 1, 'successful': 1, 'failed': 0}, '_seq_no': 1, '_primary_term': 1}


### Deleting existing model (my_model)

Delete the previous model before uploading a new one to avoid conflicts.

In [12]:
import requests
import json

# Setup connection variables
host = 'localhost'
port = 9201

# Correct API endpoint for deleting a model
delete_url = f'http://{host}:{port}/_ltr/_model/my_model_v2'

try:
    response = requests.delete(delete_url)
    result = response.json()
    
    if response.status_code == 200:
        print(f"✓ Model 'my_model_v2' deleted successfully")
        print(f"  Response: {result}")
    elif response.status_code == 404:
        print(f"ℹ Model 'my_model_v2'does not exist (nothing to delete)")
    else:
        print(f"⚠ Deletion response:")
        print(f"  Status: {response.status_code}")
        print(f"  Response: {result}")
except Exception as e:
    print(f"⚠ Warning: Could not delete model: {e}")
    print(f"  (This might be okay if the model doesn't exist)")

print("\nReady to upload new model...")

✓ Model 'my_model_v2' deleted successfully
  Response: {'_index': '.ltrstore', '_id': 'model-my_model_v2', '_version': 2, 'result': 'deleted', '_shards': {'total': 1, 'successful': 1, 'failed': 0}, '_seq_no': 2, '_primary_term': 1}

Ready to upload new model...


### Uploading Random Forest Model (RankLib Format)

Upload the Random Forest model trained with RankyMcRankFace in RankLib XML format.

In [9]:
# Upload Random Forest model (RankLib format)
import requests
import json

# Setup connection variables
host = 'localhost'
port = 9201

# Read the RankLib format Random Forest model
model_file_path = 'data/randomforest_ranklib.txt'  # RankLib XML format

try:
    with open(model_file_path, 'r') as f:
        model_definition = f.read()
    
    # Create model upload request
    create_model_request = {
        "model": {
            "name": "my_model_v2",
            "model": {
                "type": "model/ranklib",
                "definition": model_definition
            }
        }
    }
    
    upload_url = f'http://{host}:{port}/_ltr/_featureset/my_feature_set/_createmodel'
    response = requests.post(upload_url, json=create_model_request)
    result = response.json()
    
    if result.get('result') == 'created':
        print(f"✓ Random Forest model uploaded successfully!")
        print(f"  - Name: my_model_v2")
        print(f"  - Index: {result.get('_index')}")
        print(f"  - ID: {result.get('_id')}")
        print(f"  - Version: {result.get('_version')}")
    else:
        print(f"⚠ Upload response:")
        print(f"  Status: {response.status_code}")
        print(f"  Response: {result}")
except FileNotFoundError:
    print(f"❌ Error: Model file not found at {model_file_path}")
    print(f"  Make sure to run ML_Models_Training.ipynb first to generate the model")
except Exception as e:
    print(f"❌ Error uploading model: {e}")


✓ Random Forest model uploaded successfully!
  - Name: my_model_v2
  - Index: .ltrstore
  - ID: model-my_model_v2
  - Version: 1


### Uploading LambdaMART Model (RankLib Format)

Upload the Gradient Boosted (LambdaMART) model trained with RankyMcRankFace in RankLib XML format.

In [ ]:
# Upload LambdaMART model (RankLib format)
import requests
import json

# Setup connection variables
host = 'localhost'
port = 9201

# Read the RankLib format LambdaMART model
model_file_path = 'data/xgboost_ranklib.txt'  # RankLib XML format (LambdaMART)

try:
    with open(model_file_path, 'r') as f:
        model_definition = f.read()
    
    # Create model upload request
    create_model_request = {
        "model": {
            "name": "my_model_v2",
            "model": {
                "type": "model/ranklib",
                "definition": model_definition
            }
        }
    }
    
    upload_url = f'http://{host}:{port}/_ltr/_featureset/my_feature_set/_createmodel'
    response = requests.post(upload_url, json=create_model_request)
    result = response.json()
    
    if result.get('result') == 'created':
        print(f"✓ LambdaMART model uploaded successfully!")
        print(f"  - Name: my_model_xgb")
        print(f"  - Type: Gradient Boosted Ranking (LambdaMART)")
        print(f"  - Index: {result.get('_index')}")
        print(f"  - ID: {result.get('_id')}")
        print(f"  - Version: {result.get('_version')}")
    else:
        print(f"⚠ Upload response:")
        print(f"  Status: {response.status_code}")
        print(f"  Response: {result}")
except FileNotFoundError:
    print(f"❌ Error: Model file not found at {model_file_path}")
    print(f"  Make sure to run ML_Models_Training.ipynb first to generate the model")
except Exception as e:
    print(f"❌ Error uploading model: {e}")


✓ LambdaMART model uploaded successfully!
  - Name: my_model_xgb
  - Type: Gradient Boosted Ranking (LambdaMART)
  - Index: .ltrstore
  - ID: model-my_model_xgb
  - Version: 1


## Searching with a Model
Now that a model has been uploaded to Opensearch we can use it to re-rank the results of a query.


First, let's get baseline BM25 results (first-pass retrieval without LTR)

In [10]:
# Setup connection variables
host = 'localhost'
port = 9201
index_name = 'tmdb'
search_keyword = 'the dark knight'
search_url = f'http://{host}:{port}/{index_name}/_search'

# Baseline BM25 Query - pure keyword search, no LTR
bm25_query = {
    'query': {
        'multi_match': {
            'query': search_keyword,
            'fields': ['title', 'overview','release_year'],
            'lenient': True
        }
    },
    'size': 10
}

resp_bm25 = requests.post(search_url, json=bm25_query).json()

print('='*70)
print('STEP 1: BASELINE BM25 RESULTS')
print('='*70)
print(f'Query: {search_keyword}')
print(f'Total hits: {resp_bm25["hits"]["total"]["value"]}')
print('\nTop 10 Results:')
print()

bm25_results = resp_bm25['hits']['hits']
for i, hit in enumerate(bm25_results, 1):
    doc_id = hit['_source']['id']
    title = hit['_source']['title']
    score = hit['_score']
    print(f'{i:2d}. [ID: {doc_id:5s}] Score: {score:8.4f} | {title}')


STEP 1: BASELINE BM25 RESULTS
Query: the dark knight
Total hits: 627

Top 10 Results:

 1. [ID: 16234] Score:  12.7826 | Batman Beyond: Return of the Joker
 2. [ID: 13851] Score:  12.7826 | Batman: Gotham Knight
 3. [ID: 155  ] Score:  12.6779 | The Dark Knight
 4. [ID: 72003] Score:  12.6779 | The Dark Knight
 5. [ID: 268  ] Score:  12.2835 | Batman
 6. [ID: 9510 ] Score:  11.6756 | 1½ Knights - In Search of the Ravishing Princess Herzelinde
 7. [ID: 49026] Score:  10.6643 | The Dark Knight Rises
 8. [ID: 30584] Score:  10.0884 | Sword of the Valiant: The Legend of Sir Gawain and the Green Knight
 9. [ID: 5494 ] Score:   8.8945 | George and the Dragon
10. [ID: 52627] Score:   8.8464 | Knights


##  LTR with Rescore Query
Now let's use LTR properly: first get BM25 results, then rescore top N results with the LTR model.
This is the correct way to use LTR - it reranks the first-pass results, not searches the entire index.

In [11]:
# LTR Rescore Query - BM25 first-pass, then LTR rescore on top 50
ltr_rescore_query = {
    'query': {
        'multi_match': {
            'query': search_keyword,
            'fields': ['title', 'overview','release_year'],
            'lenient': True
        }
    },
    'rescore': {
        'window_size': 50,
        'query': {
            'rescore_query': {
                'sltr': {
                    'model': 'my_model_v2',
                    'params': {
                        'keywords': search_keyword
                    }
                }
            },
            'query_weight': 1.0,
            'rescore_query_weight': 2.0
        }
    },
    'size': 10
}

resp_ltr = requests.get(search_url, json=ltr_rescore_query).json()

print('='*70)
print('STEP 2: LTR RESCORE RESULTS')
print('='*70)
print(f'Query: {search_keyword}')
print('Method: BM25 first-pass + LTR rescore (window_size=50)')
print('\nTop 10 Results after Rescore:')
print()

ltr_results = resp_ltr['hits']['hits']
for i, hit in enumerate(ltr_results, 1):
    doc_id = hit['_source']['id']
    title = hit['_source']['title']
    score = hit['_score']
    print(f'{i:2d}. [ID: {doc_id:5s}] Score: {score:8.4f} | {title}')


STEP 2: LTR RESCORE RESULTS
Query: the dark knight
Method: BM25 first-pass + LTR rescore (window_size=50)

Top 10 Results after Rescore:

 1. [ID: 155  ] Score:  16.2625 | The Dark Knight
 2. [ID: 72003] Score:  16.2625 | The Dark Knight
 3. [ID: 49026] Score:  14.5266 | The Dark Knight Rises
 4. [ID: 13851] Score:  13.2313 | Batman: Gotham Knight
 5. [ID: 16234] Score:  13.1566 | Batman Beyond: Return of the Joker
 6. [ID: 268  ] Score:  12.6575 | Batman
 7. [ID: 9510 ] Score:  12.0860 | 1½ Knights - In Search of the Ravishing Princess Herzelinde
 8. [ID: 30584] Score:  10.4988 | Sword of the Valiant: The Legend of Sir Gawain and the Green Knight
 9. [ID: 29751] Score:  10.3771 | Batman Unmasked: The Psychology of the Dark Knight
10. [ID: 52627] Score:   9.9542 | Knights


## Step 3: Side-by-Side Comparison

In [12]:
print('='*100)
print('SIDE-BY-SIDE COMPARISON')
print('='*100)
print(f'Keyword: {search_keyword}\n')

print(f'{"Rank":<5} | {"BM25 Title":<40} | {"Score":<8} | {"LTR Title":<40} | {"Score":<8}')
print("-" * 108)

max_results = max(len(bm25_results), len(ltr_results))
for i in range(max_results):
    rank = i + 1
    
    if i < len(bm25_results):
        bm_title = bm25_results[i]['_source']['title'][:40]
        bm_score = f"{bm25_results[i]['_score']:.4f}"
    else:
        bm_title = "N/A"
        bm_score = "N/A"
    
    if i < len(ltr_results):
        ltr_title = ltr_results[i]['_source']['title'][:40]
        ltr_score = f"{ltr_results[i]['_score']:.4f}"
    else:
        ltr_title = "N/A"
        ltr_score = "N/A"
    
    print(f'{rank:<5} | {bm_title:<40} | {bm_score:<8} | {ltr_title:<40} | {ltr_score:<8}')

print('='*100)


SIDE-BY-SIDE COMPARISON
Keyword: the dark knight

Rank  | BM25 Title                               | Score    | LTR Title                                | Score   
------------------------------------------------------------------------------------------------------------
1     | Batman Beyond: Return of the Joker       | 12.7826  | The Dark Knight                          | 16.2625 
2     | Batman: Gotham Knight                    | 12.7826  | The Dark Knight                          | 16.2625 
3     | The Dark Knight                          | 12.6779  | The Dark Knight Rises                    | 14.5266 
4     | The Dark Knight                          | 12.6779  | Batman: Gotham Knight                    | 13.2313 
5     | Batman                                   | 12.2835  | Batman Beyond: Return of the Joker       | 13.1566 
6     | 1½ Knights - In Search of the Ravishing  | 11.6756  | Batman                                   | 12.6575 
7     | The Dark Knight Rises              

## NDCG@10 Evaluation

### What is NDCG?
NDCG (Normalized Discounted Cumulative Gain) measures ranking quality.
- Higher rank position = higher impact on score
- DCG = sum of (relevance / log2(position+1))
- NDCG = DCG / IDCG (ideal DCG with perfect ranking)

### Important: Relevance Definition
For this comparison, we need to define which documents are 'relevant' for 'rambo' query.
These IDs are examples based on the training data.
- Relevant: Documents that are actually Rambo movies
- Non-relevant: Documents about other movies

The comparison should use the SAME relevant set for both BM25 and LTR.

In [48]:
import math
import requests
from ltr.judgments import judgments_open

# ==================================================================================
# CONFIGURATION: Change this QID manually to test different queries
# ==================================================================================
qid_to_evaluate = 34

# ==================================================================================
# STEP 1: Load Judgments (ONLY keeping Grade 4 "Ideal" docs)
# ==================================================================================
qid_to_keywords = {}
qid_to_ideal_docs = {}

with judgments_open('data/title_judgments.txt') as judgment_list:
    for judgment in judgment_list:
        qid = judgment.qid
        
        if qid not in qid_to_keywords:
            qid_to_keywords[qid] = judgment_list.keywords(qid)
            qid_to_ideal_docs[qid] = set()
            
        # ONLY save the document if it has a perfect grade of 4
        if judgment.grade == 4:
            qid_to_ideal_docs[qid].add(str(judgment.docId))

# ==================================================================================
# STEP 2: Simplified NDCG Calculation
# ==================================================================================
def calculate_simple_ndcg(results, ideal_docs, k=10):
    """Calculates NDCG@k by checking if results exist in the grade-4 ideal set."""
    dcg = 0.0
    for i, hit in enumerate(results[:k]):
        doc_id = str(hit['_source']['id'])
        if doc_id in ideal_docs:
            dcg += 1.0 / math.log2(i + 2)

    idcg = 0.0
    for i in range(min(k, len(ideal_docs))):
        idcg += 1.0 / math.log2(i + 2)

    return dcg / idcg if idcg > 0 else 0.0

# ==================================================================================
# STEP 3: Execute Queries and Compare
# ==================================================================================
if qid_to_evaluate not in qid_to_keywords:
    print(f"❌ QID {qid_to_evaluate} not found in judgments.")
else:
    host = 'localhost'
    port = 9201
    index_name = 'tmdb'
    search_url = f'http://{host}:{port}/{index_name}/_search'
    
    search_keyword = qid_to_keywords[qid_to_evaluate]
    ideal_docs = qid_to_ideal_docs.get(qid_to_evaluate, set())
    
    # ---------------------------------------------------------
    # Baseline BM25 Query
    # ---------------------------------------------------------
    bm25_query = {
        'query': {
            'multi_match': {
                'query': search_keyword,
                'fields': ['title', 'overview', 'release_year'],
                'lenient': True
            }
        },
        'size': 10
    }
    
    resp_bm25 = requests.post(search_url, json=bm25_query).json()
    bm25_results = resp_bm25.get('hits', {}).get('hits', [])

    # ---------------------------------------------------------
    # LTR Rescore Query
    # ---------------------------------------------------------
    ltr_rescore_query = {
        'query': {
            'multi_match': {
                'query': search_keyword,
                'fields': ['title', 'overview', 'release_year'],
                'lenient': True
            }
        },
        'rescore': {
            'window_size': 50,
            'query': {
                'rescore_query': {
                    'sltr': {
                        'model': 'my_model_v2',
                        'params': {
                            'keywords': search_keyword
                        }
                    }
                },
                'query_weight': 1.0,
                'rescore_query_weight': 2.0
            }
        },
        'size': 10
    }
    
    # Using requests.post for both to ensure compatibility with Elasticsearch bodies
    resp_ltr = requests.post(search_url, json=ltr_rescore_query).json()
    ltr_results = resp_ltr.get('hits', {}).get('hits', [])

    # ---------------------------------------------------------
    # Calculate Metrics & Print Output
    # ---------------------------------------------------------
    bm25_ndcg = calculate_simple_ndcg(bm25_results, ideal_docs, k=10)
    ltr_ndcg = calculate_simple_ndcg(ltr_results, ideal_docs, k=10)
    
    print('='*108)
    print('SIDE-BY-SIDE COMPARISON')
    print('='*108)
    print(f'QID: {qid_to_evaluate}')
    print(f'Keyword: {search_keyword}')
    print(f'Ideal (Grade 4) Docs in Judgments: {len(ideal_docs)}')
    print(f'BM25 NDCG@10: {bm25_ndcg:.4f}  |  LTR NDCG@10: {ltr_ndcg:.4f}\n')

    print(f'{"Rank":<5} | {"BM25 Title":<40} | {"Score":<8} | {"LTR Title":<40} | {"Score":<8}')
    print("-" * 108)

    max_results = max(len(bm25_results), len(ltr_results))
    for i in range(max_results):
        rank = i + 1
        
        if i < len(bm25_results):
            bm_title = bm25_results[i]['_source']['title'][:40]
            bm_score = f"{bm25_results[i]['_score']:.4f}"
        else:
            bm_title = "N/A"
            bm_score = "N/A"
        
        if i < len(ltr_results):
            ltr_title = ltr_results[i]['_source']['title'][:40]
            ltr_score = f"{ltr_results[i]['_score']:.4f}"
        else:
            ltr_title = "N/A"
            ltr_score = "N/A"
        
        print(f'{rank:<5} | {bm_title:<40} | {bm_score:<8} | {ltr_title:<40} | {ltr_score:<8}')

    print('='*108)

Recognizing 40 queries


ConnectionError: HTTPConnectionPool(host='localhost', port=9201): Max retries exceeded with url: /tmdb/_search (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x76a36f7b6f50>: Failed to establish a new connection: [Errno 111] Connection refused'))

## Upload XGBoost Sklearn Model (my_model_v2)

Upload the XGBoost model trained with scikit-learn in JSON format to OpenSearch.

In [13]:
# Upload XGBoost sklearn model (JSON format) to OpenSearch
import requests
import json

# Setup connection variables
host = 'localhost'
port = 9201

# Read the XGBoost sklearn JSON model
model_file_path = 'data/xgboost_sklearn_opensearch.json'

try:
    with open(model_file_path, 'r') as f:
        model_request = json.load(f)
    
    # Create model upload request
    upload_url = f'http://{host}:{port}/_ltr/_featureset/my_feature_set/_createmodel'
    
    response = requests.post(upload_url, json=model_request)
    result = response.json()
    
    if result.get('result') == 'created':
        print(f"✓ XGBoost sklearn model uploaded successfully!")
        print(f"  - Name: {model_request['model']['name']}")
        print(f"  - Index: {result.get('_index')}")
        print(f"  - ID: {result.get('_id')}")
        print(f"  - Version: {result.get('_version')}")
    else:
        print(f"⚠ Upload response:")
        print(f"  Status: {response.status_code}")
        print(f"  Response: {result}")
except FileNotFoundError:
    print(f"❌ Error: Model file not found at {model_file_path}")
    print(f"  Make sure to run ML_Models_Training.ipynb first to generate the model")
except Exception as e:
    print(f"❌ Error uploading model: {e}")


⚠ Upload response:
  Status: 400
  Response: {'error': {'root_cause': [{'type': 'illegal_argument_exception', 'reason': 'Error while parsing model [xgboost_v2] with type [model/xgboost]'}], 'type': 'illegal_argument_exception', 'reason': 'Error while parsing model [xgboost_v2] with type [model/xgboost]', 'caused_by': {'type': 'illegal_argument_exception', 'reason': 'Unsupported LtrRanker format/type [model/xgboost]'}}, 'status': 400}
